# Exploratory Data Analysis — Retail Warehouse

Connects to the DuckDB warehouse and runs analytical queries.

**Prerequisites:** Run `make sample-data && make etl` first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import duckdb
import pandas as pd
import plotly.express as px

conn = duckdb.connect('../data/warehouse.duckdb', read_only=True)
print('Connected to warehouse')

In [ ]:
# Row counts per table
tables = ['customers', 'products', 'orders', 'order_items', 'payments', 'reviews', 'daily_sales']
for t in tables:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t:20s}: {n:,} rows')

In [ ]:
# Monthly revenue trend
df = conn.execute("""
    SELECT DATE_TRUNC('month', paid_at) AS month, SUM(amount) AS revenue
    FROM payments
    WHERE status = 'completed' AND paid_at IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").df()

fig = px.line(df, x='month', y='revenue', title='Monthly Revenue', markers=True)
fig.show()

In [ ]:
# Revenue by product category
df = conn.execute("""
    SELECT p.category, SUM(oi.line_total) AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY p.category ORDER BY revenue DESC
""").df()

fig = px.bar(df, x='revenue', y='category', orientation='h',
             title='Revenue by Category', color='revenue',
             color_continuous_scale='Blues')
fig.show()

In [ ]:
# Customer segment distribution
df = conn.execute("""
    SELECT segment, COUNT(*) AS count FROM customers GROUP BY segment ORDER BY count DESC
""").df()

fig = px.pie(df, values='count', names='segment', title='Customer Segments')
fig.show()

In [ ]:
# Review score distribution
df = conn.execute("""
    SELECT score, COUNT(*) AS count FROM reviews GROUP BY score ORDER BY score
""").df()

fig = px.bar(df, x='score', y='count', title='Review Score Distribution',
             color_discrete_sequence=['#2d6a9f'])
fig.show()

conn.close()